In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
from copy import deepcopy
import multiprocessing as mp
import logging
import random
from itertools import cycle
import os
from functools import partial

import mlflow
import matplotlib.pyplot as plt
from networkx import number_weakly_connected_components
import torch
import numpy as np
from dotenv import load_dotenv

from halcyon.ModelRunner import run_training_job, ModelRunner
from halcyon.Simulators import NoiseLevel, SimulationResult

from halcyon.Models import PhoenixAddAndProdLayers, PyTorchODE, HybridModel
from halcyon.Models.BaseLineModels import AlwaysDerivativeZero
from halcyon.Models.Losses import MSELossPlusMachineLearningL2


seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 

load_dotenv() 
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)


Check if MLFlow connects correctly

In [2]:
print(os.getenv("MLFLOW_TRACKING_URI"))

https://mlflow.cropresilience.org


In [3]:
[f"{experiment.name=}" for experiment in mlflow.search_experiments()]
mlflow.set_experiment("Halcyon sandbox experiment")
mlflow.set_experiment_tag("Tool name", "HALCYON") 

# Input data

## Downloading data from MLFlow

In [4]:
sim_run_id_to_use = "f5d188412b3643c0956abb757fb9eeb4"  
run = mlflow.get_run(sim_run_id_to_use)

params = run.data.params
print(params)

{'nr_genes': '20', 'sparsity': '0.1', 'num_modules': '2', 'noise_level': 'LOW', 'seed': '174'}


In [5]:
ml_flow_download_path = mlflow.artifacts.download_artifacts(run_id=sim_run_id_to_use, artifact_path="dataset")
sim_output = SimulationResult.load_from_pkl(Path(ml_flow_download_path) / "simulation_result.pkl")

2026-03-12 12:15:23,081 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,094 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,120 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,125 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,147 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,154 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,169 - WARNING - Connection pool is full, discarding connection: mlflow.cropresilience.org. Connection pool size: 10
2026-03-12 12:15:23,182 - WARNING - Connection p

## Convert sim output into datasets for use with the trainer object

In [6]:
# From sim output to datasets
single_ko_dataset, double_ko_dataset, time_scaling_factor = sim_output.create_datasets()
train_ko_dataset, validation_ko_dataset = single_ko_dataset.split_train_test(test_fraction=0.25)

train_dataset_name = "train_single_ko"
val_dataset_name ="val_single_ko"
test_dataset_name = "double_ko"
dataset_dict = {
    train_dataset_name: train_ko_dataset,
    val_dataset_name: validation_ko_dataset,
    test_dataset_name: double_ko_dataset,}

In [7]:
for name, dataset in dataset_dict.items():
    print(f"Length {name}: {len(dataset)}")

Length train_single_ko: 15
Length val_single_ko: 5
Length double_ko: 20


# Trainers (with different models) creation

In [8]:
train_jobs : list[ModelRunner] = []
device = cycle(['cuda:0', 'cuda:1'])
batch_size = 16
nr_epochs = 5

## ML model

In [9]:
for hidden_size in [10]:
    for do_only_pos in [True]:        
        ml_model = PhoenixAddAndProdLayers(
                    modelled_gene_labels=single_ko_dataset.genes,
                    hidden_size=hidden_size,
                    condition_names=single_ko_dataset.all_environments,
                    synthesis_only_positive=do_only_pos,
                    device=next(device)
            )

        # my_loss = MSELossPlusMachineLearningL2(lambda_param=0.5)
        model_runner = ModelRunner(
            model = ml_model,
            batch_size=batch_size,
            nr_epochs=nr_epochs,
            optimizer_kwargs = {'lr': 0.01}
        )
        train_jobs.append(model_runner)

## Baseline model (always derivative of 0)

In [ ]:
# baseline_model = AlwaysDerivativeZero(
#     modelled_gene_labels=single_ko_dataset.genes,
#     condition_names=single_ko_dataset.all_environments
#     )

# baseline_trainer = ModelRunner(
#     baseline_model,
#     batch_size=batch_size,
#     nr_epochs=0,
# )
# train_jobs.append(baseline_trainer)

In [10]:
for job in train_jobs:
    print(job)
    print()

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=10', 'synthesis_only_positive=True'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 nr_epochs=5,
 batch_size=16
)



In [11]:
train_jobs[0]

Modelrunner (Model=PhoenixAddAndProdLayers with hyperparams: ['device=cuda:0', 'hidden_size=10', 'synthesis_only_positive=True'],
 optimizer=Adam,
 optimizer_kwargs={'lr': 0.01},
 nr_epochs=5,
 batch_size=16
)

In [12]:
train_jobs[0].train_and_predict_for_mlflow(dataset_dict=dataset_dict, 
                                           train_data_key_in_dict=train_dataset_name,
                                           val_data_key_in_dict=val_dataset_name,
                                           test_data_key_in_dict=test_dataset_name,
                                           simulation_parent_id=sim_run_id_to_use,
                                           save_every_n_epochs=10)

2026-03-12 12:18:03,877 [MainProcess] Epoch 0, train_loss	0.17448186874389648
2026-03-12 12:18:04,414 [MainProcess] Epoch 0, val_loss	0.13972575962543488
2026-03-12 12:18:05,477 [MainProcess] Epoch 0, test_loss	0.19076785445213318
2026-03-12 12:18:06,346 [MainProcess] Epoch 1, train_loss	0.15109308063983917
2026-03-12 12:18:06,788 [MainProcess] Epoch 1, val_loss	0.12103240936994553
2026-03-12 12:18:07,861 [MainProcess] Epoch 1, test_loss	0.17405125498771667
2026-03-12 12:18:08,732 [MainProcess] Epoch 2, train_loss	0.1317521631717682
2026-03-12 12:18:09,169 [MainProcess] Epoch 2, val_loss	0.1057896763086319
2026-03-12 12:18:10,234 [MainProcess] Epoch 2, test_loss	0.15998560190200806
2026-03-12 12:18:11,085 [MainProcess] Epoch 3, train_loss	0.11585070192813873
2026-03-12 12:18:11,594 [MainProcess] Epoch 3, val_loss	0.09363760054111481
2026-03-12 12:18:12,567 [MainProcess] Epoch 3, test_loss	0.1481350213289261
2026-03-12 12:18:13,441 [MainProcess] Epoch 4, train_loss	0.10300744324922562
2

🏃 View run PhoenixAddAndProdLayers at: https://mlflow.cropresilience.org/#/experiments/3/runs/ded6730f68af4055b8a7a5b81ecf1810
🧪 View experiment at: https://mlflow.cropresilience.org/#/experiments/3


In [ ]:
from torch.nn.functional import softplus

print(softplus(train_jobs[1].model.per_gene_multipliers))

print((train_jobs[1].model.decay_params))

# Do training

In [ ]:
mp.set_start_method("spawn", force=True)
with mp.Pool(processes=4) as pool:
    pool.map(
        partial(
            run_training_job, 
            dataset_dict=dataset_dict, 
            train_data_key_in_dict=train_dataset_name, 
            simulation_parent_id=sim_run_id_to_use, 
            do_mlflow_model_logging=False,
        ),
        train_jobs
    )

# All files have now been logged to MLFlow

To see how to process this data further look into the MLFlow loading and plotting notebook(s)